# Assertions in pytest

## Mastering Assertions
---

In this lesson, you'll learn how to write **effective assertions** in pytest - the heart of every test.

You can run pytest, and now you'll learn to verify different types of results and test for exceptions.

1. [The assert Statement in pytest](#The-assert-Statement-in-pytest),
    - [Simple Assertions](#Simple-Assertions),
    - [Assertion Messages](#Assertion-Messages),
2. [Assertion Rewriting - pytest Magic](#Assertion-Rewriting---pytest-Magic),
    - [How It Works](#How-It-Works),
    - [Detailed Error Output](#Detailed-Error-Output),
3. [Comparing Different Types](#Comparing-Different-Types),
    - [Strings](#Strings),
    - [Lists](#Lists),
    - [Dictionaries](#Dictionaries),
    - [Sets](#Sets),
    - [Floating Point Numbers](#Floating-Point-Numbers),
4. [Testing Exceptions with pytest.raises](#Testing-Exceptions-with-pytest.raises),
    - [Basic Exception Testing](#Basic-Exception-Testing),
    - [Checking Exception Messages](#Checking-Exception-Messages),
    - [Testing Multiple Exceptions](#Testing-Multiple-Exceptions),
5. [Practical Example: Order Processing System](#Practical-Example:-Order-Processing-System),
6. [Common Assertion Patterns](#Common-Assertion-Patterns),
7. [🧠 EXERCISE 🧠, Test an API Response Parser](#🧠-EXERCISE-🧠,-Test-an-API-Response-Parser).

<br>

## The assert Statement in pytest

---

Unlike unittest with its many `self.assertX()` methods, pytest uses Python's built-in `assert` statement.

<br>

### Simple Assertions

---

In [ ]:
# Basic equality
assert 1 + 1 == 2

# Boolean checks
assert True
assert not False

# Comparison operators
assert 5 > 3
assert 10 <= 10
assert "hello" != "world"

# Membership
assert "a" in "abc"
assert 5 in [1, 2, 5, 10]

# Identity
assert [] is not []  # Different objects
empty_list = []
assert empty_list is empty_list  # Same object

# Truthiness
assert [1, 2, 3]  # Non-empty list is truthy
assert not []     # Empty list is falsy

<br>

### Assertion Messages

---

Add custom messages for clearer failure output:

In [ ]:
def test_user_has_permission():
    user_permissions = ["read", "write"]
    required = "delete"
    
    assert required in user_permissions, f"User lacks '{required}' permission"
    # Output: AssertionError: User lacks 'delete' permission

In [ ]:
def test_api_response_status():
    response_code = 404
    
    assert response_code == 200, f"Expected 200 OK, got {response_code}"
    # Output: AssertionError: Expected 200 OK, got 404

**Note:** Custom messages are optional - pytest's introspection usually provides enough detail.

<br>

## Assertion Rewriting - pytest Magic

---

### How It Works

---

pytest rewrites `assert` statements at import time to provide detailed failure information.

**Standard Python assert:**
```python
assert x == y
# Failure: AssertionError
```

**pytest-enhanced assert:**
```python
assert x == y
# Failure: AssertionError: assert 'hello' == 'world'
#            +  where 'hello' = get_greeting()
```

<br>

### Detailed Error Output

---

pytest shows exactly what differs:

In [ ]:
# Example test that will fail
def test_string_comparison():
    expected = "Hello, World!"
    actual = "Hello, world!"  # lowercase 'w'
    assert actual == expected

**Output:**
```
    def test_string_comparison():
        expected = "Hello, World!"
        actual = "Hello, world!"  # lowercase 'w'
>       assert actual == expected
E       AssertionError: assert 'Hello, world!' == 'Hello, World!'
E         
E         - Hello, World!
E         ?        ^
E         + Hello, world!
E         ?        ^
```

The `^` shows exactly where the difference is!

<br>

## Comparing Different Types

---

### Strings

---

In [ ]:
def format_name(first: str, last: str) -> str:
    """Format full name."""
    return f"{first.capitalize()} {last.capitalize()}"


def test_format_name():
    result = format_name("john", "doe")
    assert result == "John Doe"


def test_format_name_already_capitalized():
    result = format_name("JANE", "SMITH")
    assert result == "Jane Smith"  # capitalize() lowercases rest

**Useful string assertions:**

In [ ]:
def test_string_patterns():
    message = "Order #12345 has been shipped"
    
    # Check substring
    assert "shipped" in message
    
    # Check prefix/suffix
    assert message.startswith("Order")
    assert message.endswith("shipped")
    
    # Check pattern exists
    assert "#" in message and message.split("#")[1].split()[0].isdigit()

<br>

### Lists

---

In [ ]:
def get_active_users(users: list[dict]) -> list[str]:
    """Return usernames of active users."""
    return [u["username"] for u in users if u["active"]]


def test_get_active_users():
    users = [
        {"username": "alice", "active": True},
        {"username": "bob", "active": False},
        {"username": "charlie", "active": True},
    ]
    
    result = get_active_users(users)
    
    assert result == ["alice", "charlie"]

**When order doesn't matter:**

In [ ]:
def get_unique_tags(items: list[dict]) -> list[str]:
    """Get unique tags from items (order not guaranteed)."""
    tags = set()
    for item in items:
        tags.update(item.get("tags", []))
    return list(tags)


def test_get_unique_tags():
    items = [
        {"name": "A", "tags": ["python", "django"]},
        {"name": "B", "tags": ["django", "api"]},
    ]
    
    result = get_unique_tags(items)
    
    # Order doesn't matter - compare as sets
    assert set(result) == {"python", "django", "api"}
    # Or check length + membership
    assert len(result) == 3
    assert "python" in result

**pytest shows list differences clearly:**

```
>       assert result == ["alice", "bob", "charlie"]
E       AssertionError: assert ['alice', 'charlie'] == ['alice', 'bob', 'charlie']
E         
E         At index 1 diff: 'charlie' != 'bob'
E         Left contains one more item: 'charlie'
E         Use -v to get more diff
```

<br>

### Dictionaries

---

In [ ]:
def create_user_profile(username: str, email: str) -> dict:
    """Create a user profile dictionary."""
    return {
        "username": username,
        "email": email,
        "status": "active",
        "role": "user",
    }


def test_create_user_profile():
    profile = create_user_profile("john", "john@example.com")
    
    expected = {
        "username": "john",
        "email": "john@example.com",
        "status": "active",
        "role": "user",
    }
    
    assert profile == expected

**Partial dictionary comparison:**

In [ ]:
def test_profile_has_required_fields():
    """Test that profile contains required fields (ignoring others)."""
    profile = create_user_profile("john", "john@example.com")
    
    # Check specific keys exist
    assert "username" in profile
    assert "email" in profile
    
    # Check specific values
    assert profile["username"] == "john"
    assert profile["status"] == "active"
    
    # Check subset of keys/values
    expected_subset = {"username": "john", "status": "active"}
    assert expected_subset.items() <= profile.items()

**pytest dictionary diff output:**

```
>       assert profile == expected
E       AssertionError: assert {'email': 'john@example.com', 'role': 'user', ...} == 
E                              {'email': 'john@example.com', 'role': 'admin', ...}
E         
E         Omitting 2 identical items, use -vv to show
E         Differing items:
E         {'role': 'user'} != {'role': 'admin'}
```

<br>

### Sets

---

In [ ]:
def get_permissions(role: str) -> set[str]:
    """Get permissions for a role."""
    base = {"read"}
    
    if role == "editor":
        return base | {"write", "update"}
    elif role == "admin":
        return base | {"write", "update", "delete", "admin"}
    
    return base


def test_admin_permissions():
    permissions = get_permissions("admin")
    
    assert permissions == {"read", "write", "update", "delete", "admin"}


def test_editor_has_write_permission():
    permissions = get_permissions("editor")
    
    assert "write" in permissions
    assert "delete" not in permissions  # Editor can't delete


def test_admin_superset_of_editor():
    editor_perms = get_permissions("editor")
    admin_perms = get_permissions("admin")
    
    # Admin has all editor permissions plus more
    assert editor_perms < admin_perms  # Proper subset

<br>

### Floating Point Numbers

---

Floating point comparison is tricky due to precision:

In [ ]:
# This can fail due to floating point precision!
result = 0.1 + 0.2
print(f"0.1 + 0.2 = {result}")  # 0.30000000000000004
# assert result == 0.3  # Might fail!

**Use pytest.approx for float comparison:**

In [ ]:
import pytest


def calculate_discount(price: float, percent: float) -> float:
    """Calculate discounted price."""
    return price * (1 - percent / 100)


def test_discount_calculation():
    result = calculate_discount(100.0, 15.0)
    
    # Use pytest.approx for float comparison
    assert result == pytest.approx(85.0)


def test_floating_point_math():
    result = 0.1 + 0.2
    
    assert result == pytest.approx(0.3)


def test_approx_with_tolerance():
    """Specify custom tolerance."""
    result = 10.0
    
    # Relative tolerance (default is 1e-6)
    assert result == pytest.approx(10.001, rel=0.01)  # 1% tolerance
    
    # Absolute tolerance
    assert result == pytest.approx(10.001, abs=0.01)  # Within 0.01

<br>

## Testing Exceptions with pytest.raises

---

Good code raises exceptions for error conditions. Tests should verify this behavior.

<br>

### Basic Exception Testing

---

In [ ]:
import pytest


def divide(a: float, b: float) -> float:
    """Divide a by b."""
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b


def test_divide_by_zero_raises():
    """Test that dividing by zero raises ValueError."""
    with pytest.raises(ValueError):
        divide(10, 0)

In [ ]:
def withdraw(balance: float, amount: float) -> float:
    """Withdraw amount from balance."""
    if amount <= 0:
        raise ValueError("Amount must be positive")
    if amount > balance:
        raise ValueError("Insufficient funds")
    return balance - amount


def test_withdraw_negative_amount():
    """Test that negative withdrawal raises."""
    with pytest.raises(ValueError):
        withdraw(100, -50)


def test_withdraw_exceeds_balance():
    """Test that over-withdrawal raises."""
    with pytest.raises(ValueError):
        withdraw(100, 150)

<br>

### Checking Exception Messages

---

In [ ]:
import pytest


def test_withdraw_error_message():
    """Test the specific error message."""
    with pytest.raises(ValueError) as exc_info:
        withdraw(100, 150)
    
    # Check the exception message
    assert str(exc_info.value) == "Insufficient funds"


def test_withdraw_message_with_match():
    """Use match parameter for regex matching."""
    with pytest.raises(ValueError, match="Insufficient"):
        withdraw(100, 150)
    
    with pytest.raises(ValueError, match=r"must be.*positive"):
        withdraw(100, -10)

In [ ]:
def test_access_exception_details():
    """Access exception object for detailed assertions."""
    
    class InsufficientFundsError(Exception):
        def __init__(self, balance: float, amount: float):
            self.balance = balance
            self.amount = amount
            super().__init__(f"Cannot withdraw {amount} from {balance}")
    
    def withdraw_v2(balance: float, amount: float) -> float:
        if amount > balance:
            raise InsufficientFundsError(balance, amount)
        return balance - amount
    
    with pytest.raises(InsufficientFundsError) as exc_info:
        withdraw_v2(100, 150)
    
    # Access exception attributes
    assert exc_info.value.balance == 100
    assert exc_info.value.amount == 150

<br>

### Testing Multiple Exceptions

---

In [ ]:
import pytest


def parse_config(config_str: str) -> dict:
    """Parse config string in format 'key=value'."""
    if not config_str:
        raise ValueError("Config string cannot be empty")
    
    if "=" not in config_str:
        raise ValueError("Invalid format: missing '='")
    
    key, value = config_str.split("=", 1)
    
    if not key.strip():
        raise ValueError("Key cannot be empty")
    
    return {key.strip(): value.strip()}


def test_parse_config_empty():
    with pytest.raises(ValueError, match="cannot be empty"):
        parse_config("")


def test_parse_config_no_equals():
    with pytest.raises(ValueError, match="missing '='"):
        parse_config("invalid_config")


def test_parse_config_empty_key():
    with pytest.raises(ValueError, match="Key cannot be empty"):
        parse_config("=value")

<br>

## Practical Example: Order Processing System

---

Let's build a more realistic example with comprehensive tests.

<br>

**The code** (`src/orders.py`):

In [ ]:
# src/orders.py

from dataclasses import dataclass, field
from typing import List
from decimal import Decimal


@dataclass
class OrderItem:
    """Single item in an order."""
    product_id: str
    name: str
    price: Decimal
    quantity: int
    
    def __post_init__(self):
        if self.quantity <= 0:
            raise ValueError("Quantity must be positive")
        if self.price < 0:
            raise ValueError("Price cannot be negative")
    
    @property
    def subtotal(self) -> Decimal:
        return self.price * self.quantity


@dataclass
class Order:
    """Customer order."""
    customer_email: str
    items: List[OrderItem] = field(default_factory=list)
    discount_percent: Decimal = Decimal("0")
    
    def add_item(self, item: OrderItem) -> None:
        """Add item to order."""
        self.items.append(item)
    
    @property
    def subtotal(self) -> Decimal:
        """Total before discount."""
        return sum(item.subtotal for item in self.items)
    
    @property
    def discount_amount(self) -> Decimal:
        """Discount value."""
        return self.subtotal * self.discount_percent / 100
    
    @property
    def total(self) -> Decimal:
        """Total after discount."""
        return self.subtotal - self.discount_amount
    
    def apply_discount(self, percent: Decimal) -> None:
        """Apply percentage discount."""
        if percent < 0 or percent > 100:
            raise ValueError("Discount must be between 0 and 100")
        self.discount_percent = percent
    
    def to_dict(self) -> dict:
        """Convert to dictionary."""
        return {
            "customer_email": self.customer_email,
            "items": [
                {
                    "product_id": item.product_id,
                    "name": item.name,
                    "price": str(item.price),
                    "quantity": item.quantity,
                    "subtotal": str(item.subtotal),
                }
                for item in self.items
            ],
            "subtotal": str(self.subtotal),
            "discount_percent": str(self.discount_percent),
            "discount_amount": str(self.discount_amount),
            "total": str(self.total),
        }

<br>

**The tests** (`tests/test_orders.py`):

In [ ]:
# tests/test_orders.py

import pytest
from decimal import Decimal
from src.orders import Order, OrderItem


class TestOrderItem:
    """Tests for OrderItem."""
    
    def test_create_valid_item(self):
        """Test creating a valid order item."""
        item = OrderItem(
            product_id="PROD-001",
            name="Widget",
            price=Decimal("19.99"),
            quantity=3
        )
        
        assert item.product_id == "PROD-001"
        assert item.price == Decimal("19.99")
        assert item.quantity == 3
    
    def test_item_subtotal(self):
        """Test item subtotal calculation."""
        item = OrderItem(
            product_id="PROD-001",
            name="Widget",
            price=Decimal("10.00"),
            quantity=5
        )
        
        assert item.subtotal == Decimal("50.00")
    
    def test_zero_quantity_raises(self):
        """Test that zero quantity is rejected."""
        with pytest.raises(ValueError, match="positive"):
            OrderItem(
                product_id="PROD-001",
                name="Widget",
                price=Decimal("10.00"),
                quantity=0
            )
    
    def test_negative_price_raises(self):
        """Test that negative price is rejected."""
        with pytest.raises(ValueError, match="negative"):
            OrderItem(
                product_id="PROD-001",
                name="Widget",
                price=Decimal("-10.00"),
                quantity=1
            )


class TestOrder:
    """Tests for Order."""
    
    def test_empty_order_total(self):
        """Test that empty order has zero total."""
        order = Order(customer_email="test@example.com")
        
        assert order.total == Decimal("0")
        assert order.items == []
    
    def test_add_single_item(self):
        """Test adding one item."""
        order = Order(customer_email="test@example.com")
        item = OrderItem(
            product_id="PROD-001",
            name="Widget",
            price=Decimal("25.00"),
            quantity=2
        )
        
        order.add_item(item)
        
        assert len(order.items) == 1
        assert order.subtotal == Decimal("50.00")
        assert order.total == Decimal("50.00")
    
    def test_multiple_items(self):
        """Test order with multiple items."""
        order = Order(customer_email="test@example.com")
        
        order.add_item(OrderItem("P1", "Item 1", Decimal("10.00"), 2))  # 20.00
        order.add_item(OrderItem("P2", "Item 2", Decimal("15.00"), 3))  # 45.00
        
        assert len(order.items) == 2
        assert order.subtotal == Decimal("65.00")
    
    def test_apply_discount(self):
        """Test percentage discount."""
        order = Order(customer_email="test@example.com")
        order.add_item(OrderItem("P1", "Item", Decimal("100.00"), 1))
        
        order.apply_discount(Decimal("20"))
        
        assert order.discount_percent == Decimal("20")
        assert order.discount_amount == Decimal("20.00")
        assert order.total == Decimal("80.00")
    
    def test_invalid_discount_over_100(self):
        """Test that discount over 100% raises."""
        order = Order(customer_email="test@example.com")
        
        with pytest.raises(ValueError, match="between 0 and 100"):
            order.apply_discount(Decimal("150"))
    
    def test_invalid_discount_negative(self):
        """Test that negative discount raises."""
        order = Order(customer_email="test@example.com")
        
        with pytest.raises(ValueError):
            order.apply_discount(Decimal("-10"))
    
    def test_to_dict(self):
        """Test dictionary conversion."""
        order = Order(customer_email="test@example.com")
        order.add_item(OrderItem("P1", "Widget", Decimal("25.00"), 2))
        order.apply_discount(Decimal("10"))
        
        result = order.to_dict()
        
        assert result["customer_email"] == "test@example.com"
        assert len(result["items"]) == 1
        assert result["subtotal"] == "50.00"
        assert result["discount_percent"] == "10"
        assert result["total"] == "45.00"
        
        # Check item structure
        item_dict = result["items"][0]
        assert item_dict["product_id"] == "P1"
        assert item_dict["quantity"] == 2

<br>

## Common Assertion Patterns

---

In [ ]:
import pytest

# Pattern 1: Check type
def test_returns_correct_type():
    result = create_user_profile("john", "john@example.com")
    assert isinstance(result, dict)


# Pattern 2: Check collection length
def test_returns_expected_items():
    items = get_active_users([...])
    assert len(items) == 3


# Pattern 3: Check None
def test_returns_none_for_not_found():
    result = find_user("nonexistent")
    assert result is None


# Pattern 4: Check not None
def test_returns_result():
    result = find_user("existing")
    assert result is not None


# Pattern 5: Check any/all
def test_all_users_active():
    users = get_active_users([...])
    assert all(u.is_active for u in users)


def test_some_users_have_email():
    users = get_users()
    assert any(u.email for u in users)


# Pattern 6: Check callable
def test_callback_was_called():
    called = False
    
    def callback():
        nonlocal called
        called = True
    
    process_with_callback(callback)
    
    assert called == True

<br>

## Summary

---

| Concept | Usage | Example |
|---------|-------|----------|
| **Basic assert** | Any boolean expression | `assert x == y` |
| **Custom message** | Add context to failures | `assert x, "message"` |
| **pytest.approx** | Float comparison | `assert x == pytest.approx(1.0)` |
| **pytest.raises** | Exception testing | `with pytest.raises(ValueError):` |
| **match parameter** | Check exception message | `pytest.raises(ValueError, match="...")` |
| **exc_info.value** | Access exception details | `exc_info.value.args` |

<br>

**Key principles:**

1. **One assertion focus per test** - test one behavior, even if multiple asserts
2. **Use descriptive test names** - they're your documentation
3. **Test both happy path and error cases**
4. **Use pytest.approx for floats**
5. **Verify exception messages** - not just exception types

<br>

### 🧠 EXERCISE 🧠, Test an API Response Parser

---

You're given an API response parser. Write comprehensive tests for it.

<br>

**The code to test** (`src/api_parser.py`):

In [ ]:
# src/api_parser.py

from dataclasses import dataclass
from typing import List, Optional


@dataclass
class Product:
    """Parsed product from API."""
    id: str
    name: str
    price: float
    in_stock: bool
    categories: List[str]


def parse_product(data: dict) -> Product:
    """
    Parse product from API response.
    
    Expected format:
    {
        "id": "PROD-123",
        "name": "Product Name",
        "price": 29.99,
        "stock_status": "available" | "out_of_stock",
        "categories": ["cat1", "cat2"]
    }
    
    Raises:
        KeyError: If required field is missing
        ValueError: If data format is invalid
    """
    # Validate required fields
    required = ["id", "name", "price", "stock_status"]
    for field in required:
        if field not in data:
            raise KeyError(f"Missing required field: {field}")
    
    # Validate price
    price = data["price"]
    if not isinstance(price, (int, float)):
        raise ValueError(f"Price must be a number, got {type(price).__name__}")
    if price < 0:
        raise ValueError("Price cannot be negative")
    
    # Parse stock status
    stock_status = data["stock_status"]
    if stock_status not in ("available", "out_of_stock"):
        raise ValueError(f"Invalid stock status: {stock_status}")
    
    in_stock = stock_status == "available"
    
    # Categories are optional
    categories = data.get("categories", [])
    if not isinstance(categories, list):
        raise ValueError("Categories must be a list")
    
    return Product(
        id=data["id"],
        name=data["name"],
        price=float(price),
        in_stock=in_stock,
        categories=categories
    )

<br>

**Your task:**

Create tests for:

1. Valid product parsing (all fields)
2. Valid product without categories (optional field)
3. Product with `in_stock=False` (out_of_stock status)
4. Missing required field (KeyError)
5. Invalid price type (ValueError)
6. Negative price (ValueError)
7. Invalid stock status (ValueError)
8. Invalid categories type (ValueError)

<details>
    <summary>▶️ Solution</summary>
    
```python
# tests/test_api_parser.py

import pytest
from src.api_parser import parse_product, Product


class TestParseProductValid:
    """Tests for valid product parsing."""
    
    def test_parse_complete_product(self):
        """Test parsing product with all fields."""
        data = {
            "id": "PROD-123",
            "name": "Premium Widget",
            "price": 29.99,
            "stock_status": "available",
            "categories": ["electronics", "gadgets"]
        }
        
        product = parse_product(data)
        
        assert isinstance(product, Product)
        assert product.id == "PROD-123"
        assert product.name == "Premium Widget"
        assert product.price == pytest.approx(29.99)
        assert product.in_stock == True
        assert product.categories == ["electronics", "gadgets"]
    
    def test_parse_without_categories(self):
        """Test that categories default to empty list."""
        data = {
            "id": "PROD-456",
            "name": "Basic Widget",
            "price": 9.99,
            "stock_status": "available"
        }
        
        product = parse_product(data)
        
        assert product.categories == []
    
    def test_parse_out_of_stock(self):
        """Test parsing out of stock product."""
        data = {
            "id": "PROD-789",
            "name": "Sold Out Widget",
            "price": 49.99,
            "stock_status": "out_of_stock"
        }
        
        product = parse_product(data)
        
        assert product.in_stock == False
    
    def test_parse_integer_price(self):
        """Test that integer price is converted to float."""
        data = {
            "id": "PROD-001",
            "name": "Widget",
            "price": 100,  # Integer, not float
            "stock_status": "available"
        }
        
        product = parse_product(data)
        
        assert product.price == 100.0
        assert isinstance(product.price, float)


class TestParseProductMissingFields:
    """Tests for missing required fields."""
    
    def test_missing_id(self):
        data = {"name": "Widget", "price": 10, "stock_status": "available"}
        
        with pytest.raises(KeyError, match="id"):
            parse_product(data)
    
    def test_missing_name(self):
        data = {"id": "P1", "price": 10, "stock_status": "available"}
        
        with pytest.raises(KeyError, match="name"):
            parse_product(data)
    
    def test_missing_price(self):
        data = {"id": "P1", "name": "Widget", "stock_status": "available"}
        
        with pytest.raises(KeyError, match="price"):
            parse_product(data)
    
    def test_missing_stock_status(self):
        data = {"id": "P1", "name": "Widget", "price": 10}
        
        with pytest.raises(KeyError, match="stock_status"):
            parse_product(data)


class TestParseProductInvalidValues:
    """Tests for invalid field values."""
    
    def test_price_as_string(self):
        data = {
            "id": "P1",
            "name": "Widget",
            "price": "29.99",  # String instead of number
            "stock_status": "available"
        }
        
        with pytest.raises(ValueError, match="must be a number"):
            parse_product(data)
    
    def test_negative_price(self):
        data = {
            "id": "P1",
            "name": "Widget",
            "price": -10.00,
            "stock_status": "available"
        }
        
        with pytest.raises(ValueError, match="negative"):
            parse_product(data)
    
    def test_invalid_stock_status(self):
        data = {
            "id": "P1",
            "name": "Widget",
            "price": 10,
            "stock_status": "in_stock"  # Invalid value
        }
        
        with pytest.raises(ValueError, match="Invalid stock status"):
            parse_product(data)
    
    def test_categories_not_a_list(self):
        data = {
            "id": "P1",
            "name": "Widget",
            "price": 10,
            "stock_status": "available",
            "categories": "electronics"  # String instead of list
        }
        
        with pytest.raises(ValueError, match="must be a list"):
            parse_product(data)
```

</details>

---